# 15 · 第一个 agent：让大模型自己决定查什么

## 这一章在整个项目里的位置

```
00 总览 → 01 存档旧项目 ✅ → 02 新项目骨架 ✅ → 03 自动检查 ✅ → 04 GitHub 与 CI ✅ → 05 准备考题 ✅ → 06 评测集 ✅ → 07 判分 ✅ → 08 MLflow 评测 ✅ → 09 第一个基线 ✅ → 10 第一个工具 ✅ → 11 第二个工具 ✅ → 12 第三个工具 ✅ → 13 标准接口 MCP ✅ → 14 固定流程 ✅ → [15 第一个 agent ✅] → 16 起：改进 agent …
```

> **这一章的代码在合并请求 #17 里。** 调模型的评测已经跑过（20 道题三版、200 道题一次，合计约 ¥4–6），记在 MLflow 里。下面每一格都只读记录，**运行这本笔记不花钱**。

## 背景：固定流程做不到的事

第 14 章的固定流程，查什么、查几次都写死了，模型只能在工具挑出的 10 个候选里排序。可是有 9 道题，大模型自己就能猜对，而工具的前 30 名里根本没有正确答案：固定流程永远拿不到这 9 道。

**agent** 就是把同样的工具交给模型自己用：先查什么、查几次、要不要查一个自己想到的病、什么时候查够了，都由它决定。

计划里有个问题："为什么需要 agent？"这一章第一次用数据回答它。

## 这一章做的 6 件事

| # | 做什么 | 用的技术 | 解决什么问题 |
|---|---|---|---|
| 1 | 用标准框架搭 agent | LangChain 的 `create_agent`（基于 LangGraph） | 不自己写循环 |
| 2 | 第四个工具：按病名查编号 | rapidfuzz 模糊匹配 | agent 能查自己想到的病 |
| 3 | 看它怎么查一道题 | MLflow 的追踪记录 | 每一步都看得见 |
| 4 | 让它按规矩停下来 | 预算 + 最后一轮只能交答案 | 踩过的坑：前两版停不下来 |
| 5 | 算清花费 | 缓存折扣 | 每一轮都要重发前面的对话 |
| 6 | 大考 + 逐题比 | dev-200 + `eval compare` | agent 到底有没有更好 |

---

## 准备

运行下面这一格。它定义一个叫 `run` 的小工具，用来执行命令并显示结果。**不用看代码。**

> 运行 `uv` 的命令时，有时会先出现几行 `Building` / `Installed`：那是 uv 发现代码有变化，顺手更新了一下环境，可以忽略。

In [ ]:
import os, subprocess
from pathlib import Path
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists())
os.chdir(ROOT)
os.environ.update(NO_COLOR="1", GH_PAGER="cat", GIT_PAGER="cat")
for d in ("/opt/homebrew/bin", "/usr/local/bin", str(Path.home() / ".local/bin")):  # 编辑器不是从终端打开时，也找得到 uv
    if Path(d).is_dir() and d not in os.environ["PATH"].split(os.pathsep):
        os.environ["PATH"] += os.pathsep + d
def run(cmd, check=True):
    print(f"$ {cmd}")
    r = subprocess.run(cmd, shell=True, capture_output=True, text=True)
    print(r.stdout + r.stderr)
    if check and r.returncode != 0:
        raise RuntimeError(f"failed: {cmd}")
print("项目目录:", ROOT)

---

## 第 1 步 · 用标准框架搭 agent

### 在干什么
把模型和工具接成一个 agent。

### 背景
agent 的核心是一个循环：

```
模型读题 → 决定调用哪些工具 → 读工具的结果 → 再决定：继续查，还是交答案
            ▲                                    │
            └────────────── 继续查 ──────────────┘
```

这个循环自己写不难，难的是周边：步数上限、工具出错怎么办、答案的格式、每一步的记录。

### 用了什么技术，为什么
用 **LangChain 的 `create_agent`**（底层是 LangGraph，计划里定的框架，第 00 章）。周边的事都用它自带的"中间件"（middleware）：

| 要管的事 | 用的组件 |
|---|---|
| 每道题最多调 8 次模型 | `ModelCallLimitMiddleware`（LangChain 自带） |
| 工具报错时，把错误交给模型让它改 | `ToolErrorMiddleware`（自带），和第 13 章 MCP 的做法一样 |
| 答案按固定格式交：最多 10 个病，每个一句理由 | `ToolStrategy`（自带） |
| 最后一轮只能交答案 | 我们写的一小段中间件（第 4 步讲为什么需要它） |

工具还是那几个，在同一个进程里调用。这样考试时的"遮住"（第 10 章）对 agent 一样生效，有一条测试专门检查。第 13 章的 MCP 是给外面的应用用的。

### 公司里什么情况怎么做
- **用框架，不自己写循环**：框架把步数、出错、记录这些周边都做好了，而且团队里的人都看得懂
- **agent 的每个设置都记下来**：模型、提示词版本、预算，和评测分数放在一起

### 执行
从 MLflow 读出 agent 正式跑分时记下的设置：

In [ ]:
from raredx.config import settings
from raredx.evals.harness import Harness

harness = Harness(settings.mlflow_tracking_uri, settings.mlflow_artifact_root)
def latest(name, split="dev-200"):
    return next(r for r in harness.runs(split) if r["name"] == name and not r["code"].endswith("-dirty"))

agent_run = latest("agent")
params = harness.client.get_run(agent_run["run_id"]).data.params
for key in ("model", "thinking", "prompt_version", "max_model_calls", "tools", "knowledge_mask"):
    print(f"{key:<16} {params[key]}")

### 结果怎么看
- `max_model_calls 8`：每道题最多 8 轮
- `tools`：四个工具，比固定流程多了 `find_disease`（下一步）
- `prompt_version agent-v3`：第 3 版提示词。为什么改了两次，第 4 步讲

### 面试怎么说
> "The agent is LangChain's create_agent over the same tools as the fixed pipeline, bounded by stock middleware — a model-call budget, tool errors returned to the model, a schema for the final answer — plus one small middleware of ours that forces the answer on the last turn."

---

## 第 2 步 · 第四个工具：按病名查编号

### 在干什么
加一个工具 `find_disease`：给一个病名，找到它的 OMIM 编号。

### 背景
agent 比固定流程强的地方，是能查**自己想到的病**。可是模型记得的是病名，编号常常记错（第 09 章看到过）。没有这个工具，它想到一个病也没法用别的工具去核对。

### 用了什么技术，为什么
**rapidfuzz**：行业里常用的模糊匹配库（MIT 许可证）。病名写得不完全一样也能找到：大小写、连字符、少写几个字都没关系。在 8,478 个病名里找一次，约 20 毫秒。

### 公司里什么情况怎么做
- **给模型的工具要接得上它的习惯**：模型记名字，就给它一个按名字查的入口
- **模糊匹配可能返回好几个**（比如带编号的亚型），把名字一起给它，让它自己挑

### 执行

In [ ]:
import mlflow
from raredx.data import sources
from raredx.knowledge import hpo as hpo_knowledge
from raredx.tools.find_disease import FindDisease

registry = sources.load_sources(ROOT / "evals/datasets.toml")
hpo = hpo_knowledge.load(sources.raw_dir(registry["hpo"], ROOT / "data"))
mlflow.set_tracking_uri(settings.mlflow_tracking_uri)
mlflow.set_experiment("playground")
find_disease = FindDisease(hpo)

for name in ["Aarskog syndrome", "Noonan syndrome"]:
    print(f"查 {name!r}：")
    for m in find_disease(name, limit=4).matches:
        print(f"   {m.disease_id}  {m.disease_name:<45} 相似度 {m.similarity}")

### 结果怎么看
- `Aarskog syndrome`：正确的名字是 "Aarskog-Scott syndrome"，照样排第一
- `Noonan syndrome`：返回好几个带编号的 Noonan 综合征，模型要自己判断查哪个

### 面试怎么说
> "A model remembers diseases by name and gets their OMIM numbers wrong, so the agent has a fuzzy name lookup — without it, it could only check what the ranking proposed."

---

## 第 3 步 · 看它怎么查一道题

### 在干什么
从 MLflow 的追踪记录里，原样调出 agent 查一道题的全过程。

### 背景
这道题的正确答案是 **Noonan 综合征 1 型**。在 agent 看到的工具排名里，**根本没有这个病**：固定流程不可能答对这道题。

### 用了什么技术，为什么
第 09 章说过，一道题的每一步都记成追踪里的一层。agent 每一轮"决定调用什么"，就是模型那一层的回答。把它们按顺序列出来，就是 agent 的思路。

### 公司里什么情况怎么做
- **agent 出了问题，先看追踪记录**：它查了什么、按什么顺序、哪一步走偏了
- **上线以后也一样**：用户说"答错了"，就去翻那一次的记录

### 执行

In [ ]:
import json
from raredx.evals.harness import candidates_of

trace = next(t for t in harness.traces(agent_run["run_id"])
             if t.data.spans[0].inputs["question"]["case_id"] == "PMID_28074573_MA1")
question = trace.data.spans[0].inputs["question"]
print("病人的症状：", "、".join(question["hpo_labels"]), "\n")

turns = sorted((s for s in trace.data.spans if s.name == "Completions"), key=lambda s: s.start_time_ns)
for i, turn in enumerate(turns, 1):
    calls = turn.outputs["choices"][0]["message"].get("tool_calls") or []
    shown = []
    for c in calls:
        name, args = c["function"]["name"], json.loads(c["function"]["arguments"])
        if name == "rank_diseases":
            shown.append(f"rank_diseases（{len(args['hpo_ids'])} 个症状）")
        elif name == "find_disease":
            shown.append(f"find_disease('{args['name']}')")
        elif name in ("check_match", "lookup_disease"):
            shown.append(f"{name}({hpo.names.get(args['disease_id'], args['disease_id'])[:28]})")
        else:
            shown.append(name)
    print(f"第 {i} 轮：" + "；".join(shown))

print()
for c in candidates_of(trace.data.spans[0].outputs)[:3]:
    print(f"答案 {c.disease_name}：{c.reason}\n")

### 结果怎么看
- **第 1 轮**：用 `rank_diseases` 按 14 个症状排名
- **第 2 轮**：一次并行核对排名里的 4 个候选（Noonan 2 型、5 型、心面皮肤综合征等）。一轮里可以调用好几个工具
- **第 3 轮（关键的一步）**：它认出这些症状（眼距宽、上睑下垂、颈蹼、肺动脉瓣狭窄……）是一类病的典型表现，用 `find_disease` 按名字去查**排名里没有的** Noonan 1 型、Costello 综合征、LEOPARD 综合征、伴多发雀斑的 Noonan 综合征。这正是临床医生做鉴别诊断的思路
- **第 4、5 轮**：核对它们，查资料
- **第 6 轮**：交答案，Noonan 1 型排第 1，理由里列出了它解释的症状和遗传方式

工具的排名里没有它，是 agent 自己想到、查到、核对完的。

### 面试怎么说
> "The trace shows the agent's reasoning step by step: rank, check several candidates in parallel, then think of the RASopathies the ranking missed, find them by name and check them — and answer Noonan syndrome 1, which no fixed pipeline over the same tools could have reached."

---

## 第 4 步 · 让它按规矩停下来

### 在干什么
这一章最大的坑：agent 一开始**停不下来**。

### 背景
第 1 版在 ci-20 上试跑：20 道题只有 **4 道**交了答案。翻追踪记录发现，它一直在查。比如一道题里，它按名字查了一长串"免疫缺陷 112 型""免疫缺陷 47 型"……直到用完调用次数。

### 用了什么技术，为什么
改了两次，每次都在 ci-20 上看"交了几道"：

| 版本 | 改了什么 | 20 道题交了答案的 |
|---|---|---|
| v1 | 最多 12 轮，只在提示词里说"查够了就交答案" | 4 |
| v2 | 最多 8 轮；**最后一轮只给它"交答案"这一个工具**；提示词里写明预算 | 17 |
| v3 | 最后一轮再加一句话："预算用完了，现在交答案" | **20** |

> **一个真实踩过的坑：只给一个工具，模型照样调别的。** v2 里剩下的 3 道，最后一轮明明只给了"交答案"这个工具，DeepSeek 还是调用了前面对话里见过的其他工具。接口只检查"必须调用一个工具"，不检查是不是给它的那个。用假模型测不出这种问题，只有真模型才会犯。v3 在最后一轮明确说一句话，就解决了。

### 公司里什么情况怎么做
- **agent 的工程难点在"控制"，不在"调用"**：预算、停止条件、出错怎么办，比提示词写得漂亮更重要
- **不能只靠提示词劝**：关键的规矩用代码保证（最后一轮只给一个工具），提示词作补充
- **每改一版，先在小题集上看行为**：交了几道、平均几轮，再花钱跑大题集

### 执行
从 MLflow 读出三版在 ci-20 上的记录：

In [ ]:
for record in reversed([r for r in harness.runs("ci-20") if r["name"] == "agent"]):
    p = harness.client.get_run(record["run_id"]).data.params
    m = harness.client.get_run(record["run_id"]).data.metrics
    print(f"{p['prompt_version']}  最多 {p['max_model_calls']:>2} 轮  交了答案 {m['n_answered']:.0f}/{m['n_cases']:.0f}  第 1 名 {m['disease_top1/mean']:.0%}")

### 结果怎么看
- 三行就是上面那张表：4 → 17 → 20
- 交了答案之后，分数才有意义；v1 的第 1 名 10%，主要是因为大部分题根本没答

### 面试怎么说
> "The first agent answered 4 of 20 smoke cases: it kept investigating until its budget ran out. Prompting was not a stop rule, so the last turn now offers only the answer tool — and since the model still called tools it had seen earlier, the last request also says the budget is spent. 20 of 20 answered."

---

## 第 5 步 · 算清花费

### 在干什么
弄清 agent 一道题花多少钱。

### 背景
agent 每一轮调模型，都要把**前面的整段对话重新发一遍**：第 1 轮只有题目，第 5 轮有题目加前面所有工具的结果。所以 token 数比固定流程多得多。

可第一次试跑，命令行打印"约 $0.74"，实际只花了约 $0.08。差在哪？

### 用了什么技术，为什么
**缓存折扣**：模型服务商会记住最近发过的内容。agent 每一轮重发的前面那段对话，服务商按"缓存命中"计价，只要正常价格的 1/50（DeepSeek：每百万 token $0.006，对比 $0.30）。我们原来的估算没算这个折扣，所以高估了。现在估算时分开算。

### 公司里什么情况怎么做
- **按服务商的真实计价规则估算**：缓存命中、高峰和非高峰时段（DeepSeek 非高峰半价），都会差好几倍
- **把对话的开头保持不变**：系统提示和工具说明放在最前面且固定，缓存命中率才高

### 执行

In [ ]:
def cost(record):
    u = harness.usage(record["run_id"])
    uncached = u["usage/input_tokens"] - u["usage/cached_input_tokens"]
    usd = (uncached * settings.llm_price_input + u["usage/cached_input_tokens"] * settings.llm_price_cached_input
           + u["usage/output_tokens"] * settings.llm_price_output) / 1e6
    return u, usd

for name in ("fixed-pipeline", "agent"):
    u, usd = cost(latest(name))
    n = 200
    print(f"{name:<15} 每道题输入 {u['usage/input_tokens'] / n:>7,.0f} token（其中缓存 {u['usage/cached_input_tokens'] / u['usage/input_tokens']:.0%}）"
          f"  200 道题约 ${usd:.2f}（高峰价），${usd / 2:.2f}（非高峰）  每道题 {u['usage/seconds_per_case']:.0f} 秒")

### 结果怎么看
- **固定流程**：每道题两千多个 token，只问一次模型
- **agent**：每道题约 4 万个 token（几轮对话叠起来），大约四分之三是缓存命中
- 200 道题：agent 约 $1.2（高峰价），固定流程约 $0.26。**agent 贵 4 倍多，也慢**（每道题十几秒）

### 面试怎么说
> "An agent re-sends its whole conversation every turn — about 40,000 input tokens per case here, three quarters of them cached at a fiftieth of the price. Cost estimates must price the cache, or they overstate by an order of magnitude."

---

## 第 6 步 · 大考 + 逐题比

### 在干什么
在 dev-200 上正式考，并和固定流程逐题比。

### 背景
agent 贵 4 倍多。它必须比固定流程**明显**好，才值得。

### 执行

In [ ]:
fp_run = latest("fixed-pipeline")
for name, record in (("固定流程", fp_run), ("agent", agent_run)):
    m = harness.client.get_run(record["run_id"]).data.metrics
    print(f"{name:<6} 第 1 名 {m['disease_top1/mean']:.1%}  前 3 名 {m['disease_top3/mean']:.1%}  前 10 名 {m['disease_top10/mean']:.1%}")
print()
for k in (1, 10):
    run(f"uv run raredx eval compare {fp_run['run_id']} {agent_run['run_id']} --k {k}")

### 结果怎么看

| 系统 | 第 1 名 | 前 3 名 | 前 10 名 |
|---|---|---|---|
| 固定流程 | 35.5% | 46.5% | 54.5% |
| **agent** | **39.0%** | **47.5%** | 54.0% |

逐题比：
- **第 1 名**：只有 agent 对的 14 道，只有固定流程对的 7 道，`p = 0.19`。agent 领先，但**还不能排除是碰巧**
- **前 10 名**：10 比 11，打平

拆开看这个"打平"：
- **agent 多找到的 10 道**，正确答案**全都不在**排名工具给出的结果里：都是它自己想到、按名字查到的（像第 3 步的 Noonan）。这是固定流程永远做不到的
- **agent 丢掉的 11 道**里，有 5 道是因为它**列的答案不到 10 个**。200 道题里，它有 120 道只列了不到 10 个（中位数 8 个），前 10 名的位置没用满

### 公司里什么情况怎么做
- **诚实地报告"还没证明更好"**：p = 0.19 就是还不够，不能说成"agent 更强"
- **从拆开的结果找下一步**：agent 能找到工具找不到的病（要保留），但常常列不满 10 个（可以补）。每改一处，都在同一批题上逐题比

### 面试怎么说
> "On dev-200 the agent leads the fixed pipeline at first place, 39% to 35.5%, but paired it is 14 cases to 7, p = 0.19 — not yet established, at four times the cost. Its unique hits are all diseases it thought of itself; its losses include listing fewer than ten answers. That tells me what to change next."

---

## 本章小结

| 步骤 | 技术 | 公司里的作用 |
|---|---|---|
| 1 搭 agent | LangChain `create_agent` + 自带中间件 | 不自己写循环；预算、出错、格式都有现成组件 |
| 2 按名字查编号 | rapidfuzz | agent 能核对自己想到的病 |
| 3 看追踪记录 | MLflow 每一轮的模型回答 | agent 的思路看得见、查得到 |
| 4 按规矩停下 | 最后一轮只给交答案的工具 + 明确提醒 | 4/20 → 20/20 交答案 |
| 5 算清花费 | 缓存折扣分开计价 | 估算不再高估十倍 |
| 6 大考 + 逐题比 | dev-200 + `eval compare` | 诚实结论：领先但还没证明，贵 4 倍多 |

**一句话**：第一个 agent 能想到工具找不到的病（像 Noonan 1 型），第 1 名答对 39%；但和固定流程逐题比还没证明更好（p = 0.19），贵 4 倍多。下一步知道该改哪里了。

## 面试题

1. **How did you build the agent, and why that framework?**
   > "LangChain's create_agent, which is LangGraph underneath: the model-tool loop, middleware for a call budget, tool errors and a final-answer schema, tracing through MLflow. Nothing hand-rolled that the framework already does."

2. **What was the hardest part?**
   > "Making it stop. The first version answered 4 of 20 cases. A forced last turn — only the answer tool offered — got 17; the model still called tools it had seen, so the last request also says the budget is spent: 20 of 20. Control is the engineering problem in agents, not calling the model."

3. **Is your agent better than the fixed pipeline?**
   > "Not established. 39% against 35.5% top-1, but 14 to 7 paired is p = 0.19, and it costs four times as much. What it adds is real — every one of its unique top-10 hits is a disease the ranking never showed it — so the next iteration keeps that and fixes its short answer lists."

4. **How do you control an agent's cost?**
   > "A hard call budget enforced in middleware, compact tool outputs, a fixed prompt prefix so the provider's cache hits, and estimates that price cached tokens — 40,000 input tokens per case, three quarters cached."

**下一章 16 · 改进 agent**：保留它"能想到工具找不到的病"的长处，补上"列不满 10 个"的短处，再在同一批题上逐题比，看能不能把领先变成证明。